# NjengaData — 02 Analysis
**Mohamed Rashid | Analyst**

Kimberly built the database. My job is to ask it the right questions.

Four questions from James, four SQL queries, four findings.
Each finding gets saved as a CSV so Stacey can build the charts from it.

The queries are intentionally straightforward — no clever tricks, just SQL that does what it says.

## Setup

Connect to `njenga.db` and define a small helper function so I do not have
to type `pd.read_sql(sql, conn)` on every line. I call it `q()` — short, easy.

In [ ]:
import os
import sqlite3
import pandas as pd

# Move to project root if running from notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Connect to the database Kimberly built
conn = sqlite3.connect('data/njenga.db')

# Shortcut so we don't write pd.read_sql(..., conn) every time
def q(sql):
    return pd.read_sql(sql, conn)

print('Connected. Tables available:')
print(q("SELECT name FROM sqlite_master WHERE type='table'").to_string(index=False))

## Finding 1 — Where does James's money go?

James is planning a 2BR apartment in Nairobi. Before anything else, he needs to know
how the total cost splits across categories — how much is land, how much is labour, how much is materials.

This query pulls from the CAHF 2022 benchmark and orders categories from most to least expensive.
The second query rolls up the pre-construction costs (land, infrastructure, compliance) —
the money James spends before a single brick is laid.

In [ ]:
# Cost breakdown for a 2BR low-rise apartment in Nairobi
# Ordered from largest to smallest share
f1 = q('''
    SELECT cost_category, amount_kes, pct_of_total
    FROM cahf_cost_breakdown
    WHERE unit_type = '2BR_lowrise'
    ORDER BY pct_of_total DESC
''')
print('2BR low-rise, Nairobi — cost breakdown:')
print(f1.to_string(index=False))

In [ ]:
# Pre-construction total: what James pays before a single brick is laid
pre = q('''
    SELECT
        ROUND(SUM(pct_of_total), 1) AS pre_construction_pct,
        SUM(amount_kes)             AS pre_construction_kes
    FROM cahf_cost_breakdown
    WHERE unit_type = '2BR_lowrise'
    AND cost_category IN ('Land', 'Infrastructure', 'Compliance')
''')
print('Pre-construction share (Land + Infrastructure + Compliance):')
print(pre.to_string(index=False))

In [ ]:
f1.to_csv('data/processed/finding_1_cost_breakdown.csv', index=False)
print('Saved: finding_1_cost_breakdown.csv')

## Finding 2 — Are costs rising?

KNBS tracks construction material prices every quarter using an index where December 2019 = 100.
So if steel shows 119.6 in 2023, that means steel costs 19.6% more than it did in 2019.

The second query summarises the overall change per material across the full date range.
This is where the 19.6% steel figure comes from — confirmed with .loc[] row selection, not an estimate.

In [ ]:
# Track the three key materials quarter by quarter
f2 = q('''
    SELECT
        product, year, quarter, weight,
        index_value,
        ROUND(index_value - 100, 1) AS change_since_2019
    FROM knbs_material_index
    WHERE product IN ('Cement', 'Steel and reinforced bars', 'Timber and Wood')
    ORDER BY product, year, quarter
''')
print('Material price trends (base = 100 in Dec 2019):')
print(f2.to_string(index=False))

In [ ]:
# Overall change per material across the full date range
summary_2 = q('''
    SELECT
        product,
        weight,
        MIN(index_value) AS index_start,
        MAX(index_value) AS index_peak,
        ROUND(MAX(index_value) - MIN(index_value), 1) AS index_change,
        ROUND((MAX(index_value) - MIN(index_value)) / MIN(index_value) * 100, 1) AS pct_change
    FROM knbs_material_index
    WHERE product IN ('Cement', 'Steel and reinforced bars', 'Timber and Wood')
    GROUP BY product, weight
    ORDER BY pct_change DESC
''')
print('Total price change by material:')
print(summary_2.to_string(index=False))

In [ ]:
f2.to_csv('data/processed/finding_2_price_trends.csv', index=False)
summary_2.to_csv('data/processed/finding_2_summary.csv', index=False)
print('Saved both finding 2 files.')

## Finding 3 — Is it cheaper to build outside Nairobi?

James might consider relocating to cut costs. Integrum publishes annual regional construction
costs per square metre across three regions.

The comparison query shows 2024 figures and calculates how each region sits relative to Nairobi.
The finding is counterintuitive — the Coast is actually slightly more expensive, not cheaper.

In [ ]:
# Regional costs over time — standard bungalow (55m2)
f3 = q('''
    SELECT year, region, cost_per_m2, yoy_change_pct, total_cost_55m2
    FROM integrum_regional_costs
    WHERE cost_per_m2 IS NOT NULL
    ORDER BY year, cost_per_m2
''')
print('Regional construction costs per m2 — standard bungalow:')
print(f3.to_string(index=False))

In [ ]:
# 2024 snapshot — most complete year, all three regions
comparison = q('''
    SELECT region, cost_per_m2, total_cost_55m2
    FROM integrum_regional_costs
    WHERE year = 2024 AND cost_per_m2 IS NOT NULL
    ORDER BY cost_per_m2
''')
print('2024 comparison (55m2 bungalow):')
print(comparison.to_string(index=False))

# Show each region as % difference from Nairobi
nairobi = comparison[comparison['region'] == 'Nairobi/Mt Kenya']['cost_per_m2'].values[0]
for _, row in comparison.iterrows():
    diff = (row['cost_per_m2'] - nairobi) / nairobi * 100
    print(f"  {row['region']:22}  {diff:+.1f}% vs Nairobi")

In [ ]:
f3.to_csv('data/processed/finding_3_regional_costs.csv', index=False)
print('Saved: finding_3_regional_costs.csv')

## Finding 4 — Can James afford to build?

The affordability ratio is simple: total build cost divided by annual household income.
The result is the number of years of full income needed to pay for the build.

Build cost comes from CAHF 2022 (KES 3,015,486 for a 2BR low-rise).
Income comes from the KCHS 2022 county-level median household data.

The second block projects what that same build costs today using the KNBS materials index.
It shows what waiting actually costs in shillings.

In [ ]:
# Affordability ratio by county
# years_to_build = total build cost / annual household income
f4 = q('''
    SELECT
        county,
        median_monthly_kes      AS median_per_adult_mo,
        hh_monthly_kes,
        hh_annual_kes,
        3015486                 AS build_cost_kes,
        ROUND(3015486.0 / hh_annual_kes, 1) AS years_to_build
    FROM county_income_urban
    WHERE county IN ('Nairobi', 'Mombasa', 'Nakuru', 'Kiambu', 'Kisumu')
    AND is_national = 0
    ORDER BY years_to_build DESC
''')
print('Affordability — years of household income needed to build a 2BR:')
print(f4.to_string(index=False))

In [ ]:
# How much more will the same house cost if James waits?
# Use the KNBS materials index to project 2022 cost forward
idx_2022 = q(
    "SELECT ROUND(AVG(index_value), 2) AS idx FROM knbs_material_index "
    "WHERE category = 'Materials' AND year = 2022 AND quarter = 'Q3'"
)['idx'].values[0]

idx_now = q(
    "SELECT ROUND(AVG(index_value), 2) AS idx FROM knbs_material_index "
    "WHERE category = 'Materials' AND year = 2025 AND quarter = 'Q4'"
)['idx'].values[0]

base_cost = 3_015_486
projected = round(base_cost * (idx_now / idx_2022))
extra     = projected - base_cost

print(f'Cost of waiting (materials inflation only):')
print(f'  2022 CAHF benchmark: KES {base_cost:>12,.0f}  (index {idx_2022})')
print(f'  Projected today:     KES {projected:>12,.0f}  (index {idx_now})')
print(f'  Extra cost of delay: KES {extra:>12,.0f}')

In [ ]:
f4.to_csv('data/processed/finding_4_affordability.csv', index=False)
print('Saved: finding_4_affordability.csv')

## Summary — the four headline numbers

One cell that pulls the headline figure from each finding and prints them together.
This is what Ken reads out during the presentation.

In [ ]:
# Pull the headline number from each finding and print them together
f1_construction = q(
    "SELECT pct_of_total FROM cahf_cost_breakdown "
    "WHERE unit_type='2BR_lowrise' AND cost_category='Construction'"
)['pct_of_total'].values[0]

f2_steel = q(
    "SELECT ROUND((MAX(index_value) - MIN(index_value)) / MIN(index_value) * 100, 1) AS chg "
    "FROM knbs_material_index WHERE product = 'Steel and reinforced bars'"
)['chg'].values[0]

regions = q(
    "SELECT region, cost_per_m2 FROM integrum_regional_costs "
    "WHERE year = 2024 AND cost_per_m2 IS NOT NULL ORDER BY cost_per_m2 DESC"
)
nairobi_m2 = regions[regions['region']=='Nairobi/Mt Kenya']['cost_per_m2'].values[0]
coast_m2   = regions[regions['region']=='Coast']['cost_per_m2'].values[0]
coast_diff = round((coast_m2 - nairobi_m2) / nairobi_m2 * 100, 1)

f4_nairobi = q(
    "SELECT ROUND(3015486.0 / hh_annual_kes, 1) AS yrs "
    "FROM county_income_urban WHERE county = 'Nairobi'"
)['yrs'].values[0]

print("=" * 55)
print("  NJENGA DATA — FOUR FINDINGS")
print("=" * 55)
print(f"  F1: Construction = {f1_construction:.1f}% of a 2BR build")
print(f"  F2: Steel up {f2_steel:.1f}% since 2019")
print(f"  F3: Coast is {coast_diff:+.1f}% vs Nairobi in 2024")
print(f"  F4: Nairobi needs {f4_nairobi} years of household income")
print("=" * 55)

In [ ]:
conn.close()
print('Done. Open 03_dashboard.ipynb next.')